<a href="https://colab.research.google.com/github/41371103hjnh/114-1-/blob/main/HW1%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 匯入需要的模組
from datetime import datetime
import random
import gspread
from google.colab import auth
from google.auth import default
import pytz
import gradio as gr

# Google Colab 認證
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
taiwan_tz = pytz.timezone('Asia/Taipei')

# 定義菜單
menu = {
    "1": {"name": "醬油拉麵", "price": 130},
    "2": {"name": "味噌拉麵", "price": 150},
    "3": {"name": "豚骨拉麵", "price": 150},
    "4": {"name": "麻辣拉麵", "price": 160},
    "5": {"name": "柚香雞白湯拉麵", "price": 230}
}

# 定義運勢列表
fortune_list = [
    "大吉：運勢爆表!順利到起飛啦~",
    "吉：吉吉富吉吉!",
    "中吉：勇於嘗試會有意想不到的收穫！",
    "小吉：平淡而美好的一天~",
    "末吉：莫急莫慌莫害怕~",
    "凶：別讓挫折打倒你!",
    "大凶：來碗拉麵將逢凶化吉!"
]

def process_order(item1_qty, item2_qty, item3_qty, item4_qty, item5_qty, date_input, time_input):
    """
    處理訂單，計算總金額並回傳顯示文字與寫入試算表
    """
    now_taiwan = datetime.now(taiwan_tz)
    date = date_input if date_input else now_taiwan.strftime("%Y-%m-%d")
    time = time_input if time_input else now_taiwan.strftime("%H:%M")

    quantities = {
        "1": item1_qty, "2": item2_qty, "3": item3_qty, "4": item4_qty, "5": item5_qty,
    }

    orders = []
    total_price = 0
    order_details_text = "品項\t數量\t單價\t小計\n"

    for key, qty in quantities.items():
        if qty > 0:
            order = menu[key]
            subtotal = order["price"] * qty
            orders.append({"name": order["name"], "quantity": qty, "price": order["price"]})
            total_price += subtotal
            order_details_text += f"{order['name']}\t{qty}\t{order['price']}\t{subtotal}\n"

    if not orders:
        return "❌ 請至少點選一項餐點！", "", "", gr.Button(visible=True), gr.Button(visible=False)

    # 顯示訂單明細與總金額
    output_message = f"日期: {date}    時間: {time}\n\n"
    output_message += order_details_text
    output_message += f"\n總金額: {total_price}元"

    # 顯示隨機運勢
    fortune = random.choice(fortune_list)
    output_fortune = f"🍜 今日運勢：{fortune}"

    # 將訂單寫入 Google 試算表
    try:
        gsheets = gc.open_by_url('https://docs.google.com/spreadsheets/d/1K4Mz6mjHyLJJXb7PzLZ79kBxbW58xs4rIZ4_ZalPZj4/edit?usp=sharing')
        worksheet = gsheets.worksheet('工作表1')
        for i, order in enumerate(orders):
            is_last_item = (i == len(orders) - 1)
            subtotal = order["price"] * order["quantity"]
            row_data = [date, time, order["name"], order["quantity"], order["price"], subtotal, total_price if is_last_item else ""]
            worksheet.append_row(row_data)
        # 即使寫入成功，我們也不再回傳狀態訊息
    except gspread.exceptions.APIError as e:
        # 如果寫入失敗，可以在後台印出錯誤，但介面上不顯示
        print(f"❌ 寫入 Google 試算表時發生錯誤：{e}")

    # 回傳結果，並切換按鈕可見度
    return output_message, output_fortune, f"總金額：{total_price}元", gr.Button(visible=False), gr.Button(visible=True)


def clear_inputs():
    """清除所有輸入欄位與顯示內容，並切換按鈕可見度"""
    return (
        0, 0, 0, 0, 0, "", "", "", "", "",
        gr.Button(visible=True), gr.Button(visible=False)
    )

# 建立 Gradio 介面
with gr.Blocks(title="柯基拉麵點餐系統") as demo:
    gr.Markdown("<h1>🍜 歡迎光臨柯基拉麵 🍜</h1>")

    with gr.Row():
        with gr.Column():
            gr.Markdown("<h3>請輸入您的點餐數量：</h3>")
            item1_qty = gr.Slider(minimum=0, maximum=10, step=1, label=f"1. {menu['1']['name']} ({menu['1']['price']}元)", value=0)
            item2_qty = gr.Slider(minimum=0, maximum=10, step=1, label=f"2. {menu['2']['name']} ({menu['2']['price']}元)", value=0)
            item3_qty = gr.Slider(minimum=0, maximum=10, step=1, label=f"3. {menu['3']['name']} ({menu['3']['price']}元)", value=0)
            item4_qty = gr.Slider(minimum=0, maximum=10, step=1, label=f"4. {menu['4']['name']} ({menu['4']['price']}元)", value=0)
            item5_qty = gr.Slider(minimum=0, maximum=10, step=1, label=f"5. {menu['5']['name']} ({menu['5']['price']}元)", value=0)

            gr.Markdown("---")
            gr.Markdown("<h3>日期與時間（可選）</h3>")
            date_input = gr.Textbox(label="日期 (YYYY-MM-DD)", placeholder="留空使用今天日期")
            time_input = gr.Textbox(label="時間 (HH:MM)", placeholder="留空使用現在時間")

            order_button = gr.Button("送出訂單", visible=True)
            clear_button = gr.Button("確定", visible=False)

        with gr.Column():
            gr.Markdown("<h3>您的訂單明細：</h3>")
            output_message = gr.Textbox(label="訂單詳情", lines=8)
            output_total = gr.Textbox(label="總金額")
            output_fortune = gr.Textbox(label="今日運勢")

    # 定義按鈕行為
    order_button.click(
        fn=process_order,
        inputs=[item1_qty, item2_qty, item3_qty, item4_qty, item5_qty, date_input, time_input],
        outputs=[output_message, output_fortune, output_total, order_button, clear_button]
    )

    clear_button.click(
        fn=clear_inputs,
        inputs=[],
        outputs=[item1_qty, item2_qty, item3_qty, item4_qty, item5_qty, date_input, time_input, output_message, output_total, output_fortune, order_button, clear_button]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://729fcf46a87c0771d0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
